# 11. Preprocessing and Feature Engineering

## Objective
Build a leakage-safe, time-aware, model-ready dataset for next-day trend classification using the merged Nifty 100 financial dataset. This notebook applies minimum history filtering, target construction, lag and rolling feature engineering, feature reduction, final cleanup, and exports the final dataset.

## 1) Imports & Config

In [10]:
from pathlib import Path
from typing import Dict, List, Tuple

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 220)

MIN_HISTORY_DAYS = 250
LAG_STEPS = [1, 2, 3]
ROLL_WINDOWS = [5, 10, 20]
CORR_THRESHOLD = 0.95

ID_COLS = ["Date", "Ticker"]
PRICE_COLS = ["Open", "High", "Low", "Close", "Volume"]
RETURN_COLS = ["Return", "Log_Return", "NIFTY_RET"]
TECHNICAL_COLS = [
    "RSI", "ROC", "EMA_20", "SMA_20", "MACD", "MACD_Signal", "BB_upper", "BB_lower",
    "ATR", "Volatility_20", "Volatility_50", "Volume_MA_20", "OBV",
]
GLOBAL_COLS = ["SP500_RET", "NASDAQ_RET", "DOW_RET", "GOLD_RET", "OIL_RET", "USDINR_RET", "VIX_RET"]
GDELT_COLS = ["Event_Count", "Avg_Tone", "War_Flag", "Crisis_Flag", "Inflation_Flag", "Rate_Hike_Flag", "Recession_Flag"]
MACRO_COLS = ["Interest_Rate", "Inflation", "Unemployment", "GDP"]

## 2) Load Data

In [11]:
def resolve_input_path() -> Path:
    """Resolve merged dataset location from notebook or repo-root execution contexts."""
    candidates = [
        Path("..") / "Market_Data" / "processed" / "merged_dataset.parquet",
        Path("Market_Data") / "processed" / "merged_dataset.parquet",
        Path("ml_pipeline") / "Market_Data" / "processed" / "merged_dataset.parquet",
    ]
    for path in candidates:
        if path.exists():
            return path.resolve()
    raise FileNotFoundError("merged_dataset.parquet not found in expected locations.")


def load_dataset(path: Path) -> pd.DataFrame:
    """Load merged dataset and enforce key dtypes."""
    data = pd.read_parquet(path)
    data["Date"] = pd.to_datetime(data["Date"], errors="raise")
    data["Ticker"] = data["Ticker"].astype(str)
    return data


input_path = resolve_input_path()
df = load_dataset(input_path)

print(f"Input path: {input_path}")
print(f"Loaded shape: {df.shape}")
display(df.head(3))

Input path: C:\Users\Priyanshu\Desktop\Main\Financial-Marketing-Forecasting\ml_pipeline\Market_Data\processed\merged_dataset.parquet
Loaded shape: (65895, 41)


,Date,Ticker,Open,High,Low,Close,Volume,Return,Log_Return,RSI,ROC,EMA_20,SMA_20,MACD,MACD_Signal,BB_upper,BB_lower,ATR,Volatility_20,Volatility_50,Volume_MA_20,OBV,SP500_RET,NASDAQ_RET,DOW_RET,GOLD_RET,OIL_RET,USDINR_RET,VIX_RET,NIFTY_RET,Event_Count,Avg_Tone,War_Flag,Crisis_Flag,Inflation_Flag,Rate_Hike_Flag,Recession_Flag,Interest_Rate,Inflation,Unemployment,GDP
0,2023-03-15,ABB,3344.000000,3347.949951,3286.000000,3302.250000,228737,-0.002688,-0.002691,61.089784,2.638815,3239.739084,3245.569983,88.782405,96.142584,3429.555071,3061.584895,85.322652,0.015609,0.016247,348395.80,4383288,0.016477,0.021388,0.010568,-0.002877,-0.046390,0.005605,-0.105204,-0.006471,1201.0,-3.5560,0,0,0,0,0,4.65,301.821,3.5,27216.445
1,2023-03-16,ABB,3282.050049,3344.649902,3252.000000,3327.949951,257895,0.007783,0.007752,63.064254,0.498271,3248.140119,3254.487476,85.193424,93.952752,3436.277668,3072.697283,85.846027,0.014939,0.016254,330304.45,4641183,-0.006981,0.000516,-0.008734,0.010702,-0.052152,-0.001365,0.101559,-0.004175,1215.0,1.3026,0,0,1,0,0,4.65,301.821,3.5,27216.445
2,2023-03-17,ABB,3347.050049,3364.000000,3286.600098,3312.550049,256724,-0.004627,-0.004638,61.064636,-0.326468,3254.274398,3260.694983,80.182196,91.198641,3441.510581,3079.879385,85.242732,0.014855,0.016302,330495.80,4384459,0.017562,0.024771,0.011670,-0.003945,0.010945,0.004991,-0.120505,0.000792,1221.0,-1.4995,1,0,0,0,0,4.65,301.821,3.5,27216.445


## 3) Sanity Checks

In [12]:
def validate_and_sort(data: pd.DataFrame) -> pd.DataFrame:
    """Validate schema expectations and enforce chronological order within each ticker."""
    required_cols = set(ID_COLS + PRICE_COLS + RETURN_COLS + TECHNICAL_COLS + GLOBAL_COLS + GDELT_COLS + MACRO_COLS)
    missing = sorted(required_cols - set(data.columns))
    assert not missing, f"Missing required columns: {missing}"

    data = data.sort_values(["Ticker", "Date"]).reset_index(drop=True)
    dup_count = data.duplicated(subset=["Date", "Ticker"]).sum()
    assert dup_count == 0, f"Duplicate (Date, Ticker) rows found: {dup_count}"

    monotonic_flag = data.groupby("Ticker")["Date"].apply(lambda s: s.is_monotonic_increasing).all()
    assert bool(monotonic_flag), "Date ordering is not monotonic within at least one ticker."
    return data


df = validate_and_sort(df)

print("Sanity check summary")
print(f"Rows, Columns: {df.shape}")
print(f"Tickers: {df['Ticker'].nunique()}")
print(f"Date range: {df['Date'].min().date()} to {df['Date'].max().date()}")
print(f"Duplicate (Date, Ticker): {df.duplicated(['Date', 'Ticker']).sum()}")

dtype_summary = df.dtypes.astype(str).value_counts().rename_axis("dtype").reset_index(name="count")
display(dtype_summary)

Sanity check summary
Rows, Columns: (65895, 41)
Tickers: 100
Date range: 2023-03-15 to 2025-12-31
Duplicate (Date, Ticker): 0


,dtype,count
0,float64,32
1,int64,7
2,object,1
3,datetime64[ns],1


## 4) Minimum History Filtering

Remove tickers with insufficient historical coverage before creating features.

In [13]:
def filter_minimum_history(data: pd.DataFrame, min_days: int = 250) -> Tuple[pd.DataFrame, pd.Series]:
    """Filter out tickers with fewer than min_days observations."""
    counts = data.groupby("Ticker")["Date"].count().sort_values()
    keep_tickers = counts[counts >= min_days].index
    removed = counts[counts < min_days]
    filtered = data[data["Ticker"].isin(keep_tickers)].copy()
    filtered = filtered.sort_values(["Ticker", "Date"]).reset_index(drop=True)
    return filtered, removed


rows_before, tickers_before = len(df), df["Ticker"].nunique()
df, removed_tickers = filter_minimum_history(df, min_days=MIN_HISTORY_DAYS)
rows_after, tickers_after = len(df), df["Ticker"].nunique()

print(f"Min history threshold: {MIN_HISTORY_DAYS} days")
print(f"Tickers before -> after: {tickers_before} -> {tickers_after}")
print(f"Rows before -> after: {rows_before:,} -> {rows_after:,}")
print(f"Rows removed: {rows_before - rows_after:,} ({((rows_before - rows_after) / rows_before) * 100:.2f}%)")

print("\nRemoved tickers (<250 days):")
if removed_tickers.empty:
    print("None")
else:
    display(removed_tickers.rename("trading_days").reset_index())

Min history threshold: 250 days
Tickers before -> after: 100 -> 96
Rows before -> after: 65,895 -> 65,557
Rows removed: 338 (0.51%)

Removed tickers (<250 days):


,Ticker,trading_days
0,TMPV,3
1,TATACAP,5
2,ENRIN,83
3,HYUNDAI,247


## 5) Target Creation

Construct a next-day direction label per ticker using `shift(-1)`.

In [14]:
def create_target(data: pd.DataFrame) -> pd.DataFrame:
    """Create binary target: 1 if next close > current close, else 0, per ticker."""
    out = data.copy()
    out["next_close"] = out.groupby("Ticker", sort=False)["Close"].shift(-1)
    out["target"] = (out["next_close"] > out["Close"]).astype("Int64")
    return out


df = create_target(df)

target_valid = df["target"].dropna().astype(int)
class_dist = target_valid.value_counts().sort_index().rename_axis("target").reset_index(name="count")
class_dist["pct"] = (class_dist["count"] / class_dist["count"].sum() * 100).round(2)

print("Target class distribution (excluding last row per ticker where next_close is NaN):")
display(class_dist)

last_rows_have_no_label = df.groupby("Ticker", sort=False)["next_close"].tail(1).isna().all()
assert bool(last_rows_have_no_label), "Last row in each ticker must have NaN next_close."
assert set(target_valid.unique()).issubset({0, 1}), "Target contains unexpected values."
print("Target sanity checks passed.")

Target class distribution (excluding last row per ticker where next_close is NaN):


,target,count,pct
0,0,31791,48.49
1,1,33766,51.51


Target sanity checks passed.


## 6) Lag Features

Create lagged versions (`lag_1`, `lag_2`, `lag_3`) for returns, technical, global, GDELT, and macro columns using ticker-wise shifting.

In [15]:
def create_lag_features(data: pd.DataFrame, feature_groups: Dict[str, List[str]], lags: List[int]) -> Tuple[pd.DataFrame, List[str]]:
    """Create ticker-wise lag features with naming: feature_lag_k."""
    out = data.copy()
    created_cols: List[str] = []

    selected_features: List[str] = []
    for cols in feature_groups.values():
        for col in cols:
            if col in out.columns and col != "target":
                selected_features.append(col)

    for col in sorted(set(selected_features)):
        grouped = out.groupby("Ticker", sort=False)[col]
        for lag in lags:
            new_col = f"{col}_lag_{lag}"
            out[new_col] = grouped.shift(lag)
            created_cols.append(new_col)

    return out, created_cols


lag_feature_groups = {
    "returns": ["Return", "Log_Return", "NIFTY_RET"],
    "technical": TECHNICAL_COLS,
    "global": GLOBAL_COLS,
    "gdelt": GDELT_COLS,
    "macro": MACRO_COLS,
}

df, lag_cols = create_lag_features(df, lag_feature_groups, LAG_STEPS)
print(f"Lag features created: {len(lag_cols)}")
display(pd.Series(lag_cols[:15], name="sample_lag_columns"))

Lag features created: 102


C:\Users\Priyanshu\AppData\Local\Temp\ipykernel_14156\2287692572.py:16: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  out[new_col] = grouped.shift(lag)
C:\Users\Priyanshu\AppData\Local\Temp\ipykernel_14156\2287692572.py:16: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  out[new_col] = grouped.shift(lag)
C:\Users\Priyanshu\AppData\Local\Temp\ipykernel_14156\2287692572.py:16: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider jo

0             ATR_lag_1
1             ATR_lag_2
2             ATR_lag_3
3        Avg_Tone_lag_1
4        Avg_Tone_lag_2
5        Avg_Tone_lag_3
6        BB_lower_lag_1
7        BB_lower_lag_2
8        BB_lower_lag_3
9        BB_upper_lag_1
10       BB_upper_lag_2
11       BB_upper_lag_3
12    Crisis_Flag_lag_1
13    Crisis_Flag_lag_2
14    Crisis_Flag_lag_3
Name: sample_lag_columns, dtype: object

## 7) Rolling Features

Use leakage-safe rolling logic with `shift(1).rolling(window)` so each row only uses past values.

In [16]:
def add_rolling_features(data: pd.DataFrame, windows: List[int]) -> Tuple[pd.DataFrame, List[str]]:
    """Create ticker-wise rolling mean/std/momentum/volume stats with shift(1) to prevent leakage."""
    out = data.copy()
    added: List[str] = []

    g_ret = out.groupby("Ticker", sort=False)["Return"]
    g_close = out.groupby("Ticker", sort=False)["Close"]
    g_vol = out.groupby("Ticker", sort=False)["Volume"]

    for w in windows:
        ret_shift = g_ret.shift(1)
        close_shift = g_close.shift(1)
        vol_shift = g_vol.shift(1)

        col_ret_mean = f"return_roll_mean_{w}"
        col_ret_std = f"return_roll_std_{w}"
        col_mom = f"momentum_{w}"
        col_vol_mean = f"volume_roll_mean_{w}"
        col_vol_std = f"volume_roll_std_{w}"

        out[col_ret_mean] = ret_shift.groupby(out["Ticker"], sort=False).rolling(w).mean().reset_index(level=0, drop=True)
        out[col_ret_std] = ret_shift.groupby(out["Ticker"], sort=False).rolling(w).std().reset_index(level=0, drop=True)
        out[col_mom] = close_shift / close_shift.groupby(out["Ticker"], sort=False).shift(w - 1) - 1
        out[col_vol_mean] = vol_shift.groupby(out["Ticker"], sort=False).rolling(w).mean().reset_index(level=0, drop=True)
        out[col_vol_std] = vol_shift.groupby(out["Ticker"], sort=False).rolling(w).std().reset_index(level=0, drop=True)

        added.extend([col_ret_mean, col_ret_std, col_mom, col_vol_mean, col_vol_std])

    return out, added


df, rolling_cols = add_rolling_features(df, ROLL_WINDOWS)
print(f"Rolling features created: {len(rolling_cols)}")
display(pd.Series(rolling_cols, name="rolling_columns"))

Rolling features created: 15


0      return_roll_mean_5
1       return_roll_std_5
2              momentum_5
3      volume_roll_mean_5
4       volume_roll_std_5
5     return_roll_mean_10
6      return_roll_std_10
7             momentum_10
8     volume_roll_mean_10
9      volume_roll_std_10
10    return_roll_mean_20
11     return_roll_std_20
12            momentum_20
13    volume_roll_mean_20
14     volume_roll_std_20
Name: rolling_columns, dtype: object

## 8) Feature Reduction

Apply domain-aware pruning and high-correlation reduction (`threshold = 0.95`) without blind dropping.

In [17]:
def feature_reduction(data: pd.DataFrame, corr_threshold: float = 0.95) -> Tuple[pd.DataFrame, List[str], pd.DataFrame]:
    """Drop redundant/correlated features with domain-aware rules and diagnostics."""
    out = data.copy()
    drops: List[str] = []

    # 1) Redundant transform: keep Return, drop Log_Return.
    if "Log_Return" in out.columns:
        drops.append("Log_Return")

    # 2) Global indices multicollinearity: prefer SP500_RET as representative.
    global_core = [c for c in ["SP500_RET", "NASDAQ_RET", "DOW_RET"] if c in out.columns]
    global_corr = pd.DataFrame()
    if len(global_core) >= 2:
        global_corr = out[global_core].corr().abs()
        if "SP500_RET" in global_core:
            for c in ["NASDAQ_RET", "DOW_RET"]:
                if c in global_core and global_corr.loc["SP500_RET", c] >= corr_threshold:
                    drops.append(c)

    # 3) Macro multicollinearity: priority keep order.
    macro_keep_priority = ["Interest_Rate", "Inflation", "Unemployment", "GDP"]
    macro_present = [c for c in macro_keep_priority if c in out.columns]
    macro_corr = pd.DataFrame()
    if len(macro_present) >= 2:
        macro_corr = out[macro_present].corr().abs()
        for i, left in enumerate(macro_present):
            for right in macro_present[i + 1:]:
                if macro_corr.loc[left, right] >= corr_threshold:
                    if right not in drops:
                        drops.append(right)

    drops = sorted(set(drops))
    out = out.drop(columns=[c for c in drops if c in out.columns])

    diagnostics = pd.DataFrame({
        "dropped_feature": drops,
        "reason": [
            "Redundant with Return" if c == "Log_Return" else
            "High corr with SP500_RET" if c in {"NASDAQ_RET", "DOW_RET"} else
            "High macro multicollinearity" for c in drops
        ],
    })
    return out, drops, diagnostics


rows_pre_reduction, cols_pre_reduction = df.shape
df, dropped_features, drop_diagnostics = feature_reduction(df, corr_threshold=CORR_THRESHOLD)
assert "target" in df.columns, "'target' column missing after feature reduction. Re-run notebook from the target creation step or Run All Cells in order."
rows_post_reduction, cols_post_reduction = df.shape

print(f"Columns before reduction: {cols_pre_reduction}")
print(f"Columns after reduction: {cols_post_reduction}")
print(f"Dropped columns: {len(dropped_features)}")
display(drop_diagnostics if not drop_diagnostics.empty else pd.DataFrame({"message": ["No features dropped by reduction rules."]}))

Columns before reduction: 160
Columns after reduction: 157
Dropped columns: 3


,dropped_feature,reason
0,GDP,High macro multicollinearity
1,Log_Return,Redundant with Return
2,NASDAQ_RET,High corr with SP500_RET


## 9) Final Cleanup

Drop rows with NaNs induced by lag/rolling/target helper columns, remove helpers, and assert final data integrity.

In [18]:
def final_cleanup(data: pd.DataFrame) -> pd.DataFrame:
    """Drop engineered NaNs, helper columns, and enforce strict integrity assertions."""
    out = data.copy()

    helper_cols = ["next_close"]
    existing_helpers = [c for c in helper_cols if c in out.columns]

    rows_before = len(out)
    out = out.dropna(axis=0).copy()
    rows_after_na = len(out)

    if existing_helpers:
        out = out.drop(columns=existing_helpers)

    out = out.sort_values(["Ticker", "Date"]).reset_index(drop=True)

    # Assertions
    assert out.isna().sum().sum() == 0, "NaNs present after cleanup."
    assert out.duplicated(subset=["Date", "Ticker"]).sum() == 0, "Duplicate (Date, Ticker) detected after cleanup."
    assert out.groupby("Ticker")["Date"].apply(lambda s: s.is_monotonic_increasing).all(), "Date order broken after cleanup."
    if "target" not in out.columns:
        raise AssertionError("'target' column is missing before final cleanup. Execute target creation and downstream feature cells in order.")
    assert out["target"].isin([0, 1]).all(), "Target contains invalid values after cleanup."

    print(f"Rows before cleanup: {rows_before:,}")
    print(f"Rows after dropna: {rows_after_na:,}")
    print(f"Rows dropped in cleanup: {rows_before - rows_after_na:,}")
    return out


df_final = final_cleanup(df)
print(f"Final cleaned shape: {df_final.shape}")

Rows before cleanup: 65,557
Rows after dropna: 63,541
Rows dropped in cleanup: 2,016
Final cleaned shape: (63541, 156)


## 10) Save Dataset

Persist final model dataset to the processed folder.

In [24]:
def resolve_output_path() -> Path:
    """Resolve final output path from notebook or repo-root execution contexts."""
    candidates = [
        Path("..") / "Market_Data" / "processed" / "final_model_dataset.parquet",
        Path("Market_Data") / "processed" / "final_model_dataset.parquet",
        Path("ml_pipeline") / "Market_Data" / "processed" / "final_model_dataset.parquet",
    ]
    return candidates[0]


output_path = resolve_output_path()
output_path.parent.mkdir(parents=True, exist_ok=True)
df_final.to_parquet(output_path, index=False)
## save to csv as well for easier inspection and compatibility with other tools
df_final.to_csv(output_path.with_suffix(".csv"), index=False)

assert output_path.exists(), "Output parquet file was not created."
print(f"Saved final dataset: {output_path.resolve()}")

Saved final dataset: C:\Users\Priyanshu\Desktop\Main\Financial-Marketing-Forecasting\ml_pipeline\Market_Data\processed\final_model_dataset.parquet


## 11) Leakage Checklist (Mandatory Verification)

Explicitly verify leakage constraints before training.

In [25]:
def leakage_checklist(data_before_cleanup: pd.DataFrame, data_final: pd.DataFrame) -> pd.DataFrame:
    """Return explicit leakage and integrity checks as a report table."""
    checks = []

    # Target uses future only for label.
    future_label_rule = data_before_cleanup.groupby("Ticker", sort=False)["Close"].shift(-1).equals(data_before_cleanup["next_close"])
    checks.append(("Target uses future only for label", bool(future_label_rule)))

    # Rolling features must be shifted.
    rolling_cols_present = [c for c in data_before_cleanup.columns if c.startswith("return_roll_") or c.startswith("momentum_") or c.startswith("volume_roll_")]
    checks.append(("Rolling features created with shift(1)", len(rolling_cols_present) > 0))

    # No global operations across tickers in transformations (enforced by groupby design).
    checks.append(("Feature transforms grouped by Ticker", True))

    # No future leakage in macro/global features (lagged by construction in merge, plus ticker-wise lags added).
    macro_global_lags = [c for c in data_final.columns if any(c.startswith(base + "_lag_") for base in GLOBAL_COLS + MACRO_COLS)]
    checks.append(("Macro/global lag features are past-only", len(macro_global_lags) > 0))

    # Chronological order check.
    chrono_ok = data_final.groupby("Ticker")["Date"].apply(lambda s: s.is_monotonic_increasing).all()
    checks.append(("Dataset remains time-ordered", bool(chrono_ok)))

    # No duplicates and no missing.
    checks.append(("No duplicate (Date, Ticker)", data_final.duplicated(["Date", "Ticker"]).sum() == 0))
    checks.append(("No missing values", data_final.isna().sum().sum() == 0))

    report = pd.DataFrame(checks, columns=["check", "passed"])
    return report


leakage_report = leakage_checklist(df, df_final)
display(leakage_report)
assert leakage_report["passed"].all(), "One or more leakage/integrity checks failed."

,check,passed
0,Target uses future only for label,True
1,Rolling features created with shift(1),True
2,Feature transforms grouped by Ticker,True
3,Macro/global lag features are past-only,True
4,Dataset remains time-ordered,True
5,"No duplicate (Date, Ticker)",True
6,No missing values,True


## 12) Final Summary

In [26]:
def summarize_final_dataset(data: pd.DataFrame) -> None:
    """Print final dataset summary for downstream modeling readiness."""
    print("Final dataset summary")
    print("-" * 60)
    print(f"Final shape: {data.shape}")
    print(f"Feature count (excluding target): {data.shape[1] - 1}")
    print(f"Ticker count: {data['Ticker'].nunique()}")
    print(f"Date range: {data['Date'].min().date()} to {data['Date'].max().date()}")

    target_dist = data["target"].value_counts(normalize=True).sort_index() * 100
    print("Target distribution (%):")
    for cls, pct in target_dist.items():
        print(f"  class {int(cls)}: {pct:.2f}%")

    print("\nSample rows:")
    display(data.head(3))


summarize_final_dataset(df_final)

Final dataset summary
------------------------------------------------------------
Final shape: (63541, 156)
Feature count (excluding target): 155
Ticker count: 96
Date range: 2023-04-18 to 2025-12-30
Target distribution (%):
  class 0: 48.59%
  class 1: 51.41%

Sample rows:


,Date,Ticker,Open,High,Low,Close,Volume,Return,RSI,ROC,EMA_20,SMA_20,MACD,MACD_Signal,BB_upper,BB_lower,ATR,Volatility_20,Volatility_50,Volume_MA_20,OBV,SP500_RET,DOW_RET,GOLD_RET,OIL_RET,USDINR_RET,VIX_RET,NIFTY_RET,Event_Count,Avg_Tone,War_Flag,Crisis_Flag,Inflation_Flag,Rate_Hike_Flag,Recession_Flag,Interest_Rate,Inflation,Unemployment,target,ATR_lag_1,ATR_lag_2,ATR_lag_3,Avg_Tone_lag_1,Avg_Tone_lag_2,Avg_Tone_lag_3,BB_lower_lag_1,BB_lower_lag_2,BB_lower_lag_3,BB_upper_lag_1,BB_upper_lag_2,BB_upper_lag_3,Crisis_Flag_lag_1,Crisis_Flag_lag_2,Crisis_Flag_lag_3,DOW_RET_lag_1,DOW_RET_lag_2,DOW_RET_lag_3,EMA_20_lag_1,EMA_20_lag_2,EMA_20_lag_3,Event_Count_lag_1,Event_Count_lag_2,Event_Count_lag_3,GDP_lag_1,GDP_lag_2,GDP_lag_3,GOLD_RET_lag_1,GOLD_RET_lag_2,GOLD_RET_lag_3,Inflation_lag_1,Inflation_lag_2,Inflation_lag_3,Inflation_Flag_lag_1,Inflation_Flag_lag_2,Inflation_Flag_lag_3,Interest_Rate_lag_1,Interest_Rate_lag_2,Interest_Rate_lag_3,Log_Return_lag_1,Log_Return_lag_2,Log_Return_lag_3,MACD_lag_1,MACD_lag_2,MACD_lag_3,MACD_Signal_lag_1,MACD_Signal_lag_2,MACD_Signal_lag_3,NASDAQ_RET_lag_1,NASDAQ_RET_lag_2,NASDAQ_RET_lag_3,NIFTY_RET_lag_1,NIFTY_RET_lag_2,NIFTY_RET_lag_3,OBV_lag_1,OBV_lag_2,OBV_lag_3,OIL_RET_lag_1,OIL_RET_lag_2,OIL_RET_lag_3,ROC_lag_1,ROC_lag_2,ROC_lag_3,RSI_lag_1,RSI_lag_2,RSI_lag_3,Rate_Hike_Flag_lag_1,Rate_Hike_Flag_lag_2,Rate_Hike_Flag_lag_3,Recession_Flag_lag_1,Recession_Flag_lag_2,Recession_Flag_lag_3,Return_lag_1,Return_lag_2,Return_lag_3,SMA_20_lag_1,SMA_20_lag_2,SMA_20_lag_3,SP500_RET_lag_1,SP500_RET_lag_2,SP500_RET_lag_3,USDINR_RET_lag_1,USDINR_RET_lag_2,USDINR_RET_lag_3,Unemployment_lag_1,Unemployment_lag_2,Unemployment_lag_3,VIX_RET_lag_1,VIX_RET_lag_2,VIX_RET_lag_3,Volatility_20_lag_1,Volatility_20_lag_2,Volatility_20_lag_3,Volatility_50_lag_1,Volatility_50_lag_2,Volatility_50_lag_3,Volume_MA_20_lag_1,Volume_MA_20_lag_2,Volume_MA_20_lag_3,War_Flag_lag_1,War_Flag_lag_2,War_Flag_lag_3,return_roll_mean_5,return_roll_std_5,momentum_5,volume_roll_mean_5,volume_roll_std_5,return_roll_mean_10,return_roll_std_10,momentum_10,volume_roll_mean_10,volume_roll_std_10,return_roll_mean_20,return_roll_std_20,momentum_20,volume_roll_mean_20,volume_roll_std_20
0,2023-04-18,ABB,3240.0,3259.850098,3202.899902,3207.699951,277893,-0.006596,41.750365,-5.418037,3293.842277,3323.947485,0.773588,28.965845,3451.152204,3196.742767,81.034466,0.015720,0.017148,347703.75,5419561,0.003306,0.002972,-0.003996,-0.020480,0.002009,-0.007030,-0.006796,1177.0,-1.8486,0,0,0,0,0,4.83,302.845,3.4,0,82.887102,84.878417,84.972919,1.5097,-1.0067,-1.2784,3212.558229,3225.590969,3234.815401,3444.791746,3439.973997,3435.194560,0.0,0.0,0.0,-0.004209,-0.001137,0.002926,3302.909891,3310.689879,3319.183551,794.0,1162.0,1208.0,27530.055,27530.055,27530.055,-0.019155,0.003043,0.007893,302.845,302.845,302.845,0.0,0.0,0.0,4.83,4.83,4.83,-0.000310,0.008223,-0.022806,8.797362,16.741922,26.536732,36.013909,42.818046,49.337077,-0.003518,-0.008522,-0.004343,0.000000,0.005084,0.005575,5697454.0,5983513.0,5580538.0,0.004382,0.021219,0.022448,-3.284259,-2.845458,-4.277345,43.643701,43.730154,40.852504,0.0,0.0,0.0,0.0,0.0,0.0,-0.000310,0.008256,-0.022548,3328.674988,3332.782483,3335.004980,-0.002069,-0.004135,-0.000041,-0.003415,0.001051,0.001210,3.4,3.4,3.4,-0.041011,-0.000524,0.006853,0.015675,0.015917,0.017100,0.017117,0.017112,0.017525,345245.95,347485.65,341155.70,0.0,0.0,0.0,-0.009963,0.018248,-0.049637,374862.4,133157.462505,-0.003228,0.015228,-0.047900,451317.3,258145.709730,-0.001138,0.015675,-0.022182,345245.95,212919.005967
1,2023-04-19,ABB,3213.0,3243.000000,3186.050049,3201.050049,266982,-0.002073,41.150164,-4.876450,3285.004922,3317.602490,-6.052138,21.962249,3455.579143,3179.625838,79.314143,0.015573,0.016910,348158.10,5152579,0.000855,-0.000310,0.006619,0.000371,0.001941,-0.007080,-0.002637,1245.0,-2.6255,0,0,0,0,0,4.83,302.845,3.4,0,81.034466,82.887102,84.878417,-1.8486,1.5097,-1.0067,3196.742767,3212.558229,3225.590969,3451.152204,3444.791746,3439.